# v4 Model Test & Evaluation

Load the **best finished checkpoint** (`final.pt` preferred, else `last.pt` by lower val loss), then run **step-by-step metrics**.

Each step: **what it measures → why we use it → good/bad range → link**.

At the end: what matters most now, likely bottleneck (architecture / data / hardware), and how to push quality higher.

> Run top → bottom. First cells load model; later cells only evaluate.


## 0 — Setup

Imports from `transformer.py` (same code that trained v4). Change `PREFIX` only if you trained with another file prefix.


In [1]:
# Cell 0: paths + imports
import os, math, warnings, collections, re
from pathlib import Path
from os.path import exists

import torch
import pandas as pd
import numpy as np
from IPython.display import display

warnings.filterwarnings("ignore")

# Always work from this folder
ROOT = Path("/home/creator/Desktop/Deeplearning/own_trainied_models/Base_model_version/v4").resolve()
os.chdir(ROOT)
print("cwd:", os.getcwd())

# Match training mode / checkpoint prefix — set BEFORE importing transformer
TRAIN_MODE = "local_v4"
PREFIX = "v4_local_mixchat_"
os.environ["RUN_MODE"] = TRAIN_MODE

# How many val batches / decode examples (raise for more stable numbers)
MAX_ACC_BATCHES = 60
N_DECODE = 40
N_SHOW = 12

print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

cwd: /home/creator/Desktop/Deeplearning/own_trainied_models/Base_model_version/v4
torch: 2.11.0+cu128
cuda: True
gpu: NVIDIA GeForce GTX 1650


In [2]:
# Cell 1: import training module APIs
import transformer as T

# Keep module mode in sync (in case env was set after a prior import in this kernel)
T.RUN_MODE = TRAIN_MODE
CFG = T.get_run_config()
print("config:", CFG)

True 1
NVIDIA GeForce GTX 1650


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login: OK
Number display: normal decimals (not scientific like 1.1e-05)
config: {'mode': 'local_v4', 'train_pairs': None, 'valid_pairs': None, 'batch_size': 6, 'num_epochs': 5, 'accum_iter': 6, 'base_lr': 1.0, 'max_padding': 64, 'warmup': 3000, 'N': 4, 'd_model': 512, 'd_ff': 2048, 'h': 8, 'dropout': 0.1, 'file_prefix': 'v4_local_mixchat_', 'vocab_path': 'v4_vocab_local_mixchat.json', 'save_every_steps': 2000, 'train_sources': ['dailydialog', 'alpaca'], 'valid_sources': ['dailydialog'], 'alpaca_train_pairs': 52000, 'data_seed': 42}


## 1 — Pick best checkpoint (`final` vs `last`)

**Why:** Training can stop mid-epoch. `final.pt` = finished run. `last.pt` = latest save.  
**Rule here:** prefer `final` if present; if both exist, pick **lower `val_loss`**.

**Good:** lower val loss over epochs (your final ~3.06).  
**Bad:** rising val loss while train falls → overfitting.

- [Overfitting / generalization](https://developers.google.com/machine-learning/crash-course/generalization/video-lecture)
- [Checkpointing practice](https://pytorch.org/tutorials/recipes/recipes/saving_and_loading_a_general_checkpoint.html)


In [3]:
# Cell 2: choose best of final / last
final_p = ROOT / f"{PREFIX}final.pt"
last_p = ROOT / f"{PREFIX}last.pt"

def peek_meta(path):
    if not path.exists():
        return None
    obj = torch.load(path, map_location="cpu")
    if not isinstance(obj, dict):
        return {"path": str(path), "val_loss": None, "epoch": None, "step": None, "raw": True}
    return {
        "path": str(path),
        "val_loss": obj.get("val_loss"),
        "epoch": obj.get("epoch"),
        "step": obj.get("step"),
        "epoch_rows": obj.get("epoch_rows"),
        "config": obj.get("config"),
        "obj": obj,
    }

cands = [m for m in (peek_meta(final_p), peek_meta(last_p)) if m is not None]
assert cands, f"No checkpoints found for prefix {PREFIX}"

# Prefer final; if both have val_loss, pick lower
def score(m):
    is_final = m["path"].endswith("final.pt")
    vl = m["val_loss"]
    # lower val_loss better; final gets tiny bonus so equal loss prefers final
    if vl is None:
        return (0 if is_final else 1, 999.0)
    return (0, float(vl) - (0.0001 if is_final else 0.0))

best = sorted(cands, key=score)[0]
CKPT_PATH = best["path"]
CKPT = best["obj"]
print("Available:")
for m in cands:
    print(f"  {Path(m['path']).name}: val_loss={m['val_loss']} epoch={m['epoch']} step={m['step']}")
print("\nSELECTED:", Path(CKPT_PATH).name)
print("val_loss:", CKPT.get("val_loss"), "| epoch:", CKPT.get("epoch"), "| step:", CKPT.get("step"))

# Use config stored in checkpoint (truth for architecture)
if isinstance(CKPT.get("config"), dict):
    CFG = dict(CKPT["config"])
    print("Using checkpoint config keys:", sorted(CFG.keys()))

Available:
  v4_local_mixchat_final.pt: val_loss=3.057927370071411 epoch=4 step=120638
  v4_local_mixchat_last.pt: val_loss=3.057927370071411 epoch=4 step=120638

SELECTED: v4_local_mixchat_final.pt
val_loss: 3.057927370071411 | epoch: 4 | step: 120638
Using checkpoint config keys: ['N', 'accum_iter', 'alpaca_train_pairs', 'base_lr', 'batch_size', 'd_ff', 'd_model', 'data_seed', 'dropout', 'file_prefix', 'h', 'max_padding', 'mode', 'num_epochs', 'save_every_steps', 'train_pairs', 'train_sources', 'valid_pairs', 'valid_sources', 'vocab_path', 'warmup']


## 2 — Load vocab + validation data

Same word-level vocab + DailyDialog valid split used in training.  
Without matching vocab, metrics are meaningless.


In [4]:
# Cell 3: tokenizer, valid pairs, vocab
spacy_en = T.load_tokenizer_en()
train_pairs, valid_pairs = T.build_train_valid_pairs(CFG)
vocab_path = CFG.get("vocab_path", f"{PREFIX.replace('_','')}vocab.json")
if not exists(vocab_path):
    vocab_path = "v4_vocab_local_mixchat.json"
vocab = T.load_vocab(spacy_en, train_pairs, vocab_path=vocab_path)

print(f"train={len(train_pairs)} valid={len(valid_pairs)} vocab={vocab.get_vocab_size()}")
print("specials:", {t: vocab.token_to_id(t) for t in ['<s>','</s>','<blank>','<unk>'] if vocab.token_to_id(t) is not None})

Data sources -> train: dailydialog,alpaca | valid: dailydialog
Finished.
Vocabulary size:
30000
train=128026 valid=7069 vocab=30000
specials: {'<s>': 0, '</s>': 1, '<blank>': 2, '<unk>': 3}


## 3 — Rebuild model + load weights

Architecture must match checkpoint (`N`, `d_model`, `h`, `d_ff`). Wrong size → load error or silent garbage.


In [ ]:
# Cell 4: build model and load weights
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = T.make_model(
    vocab.get_vocab_size(),
    vocab.get_vocab_size(),
    N=int(CFG["N"]),
    d_model=int(CFG["d_model"]),
    d_ff=int(CFG.get("d_ff", 2048)),
    h=int(CFG.get("h", 8)),
    dropout=float(CFG.get("dropout", 0.1)),
)
T.load_model_weights(model, CKPT_PATH, map_location="cpu")
model = model.to(device).eval()

n_params = sum(p.numel() for p in model.parameters())
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"device={device}  params={n_params:,}  trainable={n_train:,}")
print(f"N={CFG['N']} d_model={CFG['d_model']} h={CFG.get('h')} d_ff={CFG.get('d_ff')} max_pad={CFG['max_padding']}")

pad_idx = vocab.token_to_id("<blank>")
bos_idx = vocab.token_to_id("<s>")
eos_idx = vocab.token_to_id("</s>")

# dataloaders for metrics
_, valid_loader = T.create_dataloaders(
    device, vocab, spacy_en, valid_pairs, valid_pairs,
    batch_size=int(CFG["batch_size"]), max_padding=int(CFG["max_padding"]),
)
_, single_loader = T.create_dataloaders(
    device, vocab, spacy_en, valid_pairs, valid_pairs,
    batch_size=1, max_padding=int(CFG["max_padding"]),
)

## 4 — Training history (old → new view)

**What:** train/val loss per epoch from the checkpoint.  
**Why:** First check before fancy metrics — did learning actually happen?  
**Era:** Classic ML / early DL (pre-LLM era still the first gate).

**Good:** both losses fall; gap small.  
**Bad:** train << val and gap growing.

- [Loss curves guide](https://developers.google.com/machine-learning/testing-debugging/metrics/interpreting-loss)


In [ ]:
# Cell 5: epoch history table + simple plot
rows = list(CKPT.get("epoch_rows") or [])
hist = pd.DataFrame(rows)
if len(hist):
    # tokens may be tensors
    if "tokens" in hist.columns:
        hist["tokens"] = hist["tokens"].apply(lambda x: int(x) if hasattr(x, "item") else int(x))
    display(hist)
    try:
        import matplotlib.pyplot as plt
        fig, ax = plt.subplots(figsize=(7, 3.5))
        ax.plot(hist["epoch"], hist["train_loss"], "o-", label="train_loss")
        ax.plot(hist["epoch"], hist["val_loss"], "s-", label="val_loss")
        ax.set_xlabel("epoch"); ax.set_ylabel("loss"); ax.legend(); ax.set_title("v4 loss by epoch")
        ax.grid(True, alpha=0.3)
        plt.show()
    except Exception as e:
        print("plot skipped:", e)
else:
    print("No epoch_rows in checkpoint.")

VAL_LOSS = CKPT.get("val_loss")
print("checkpoint val_loss:", VAL_LOSS)

## 5 — Perplexity (PPL)

**Formula:** `PPL = exp(val_loss)` (for mean NLL).  
**Why:** Interprets loss as “effective branching factor” — how surprised the model is.  
**Era:** Language modeling since early 90s; still used by OpenAI / Google / Meta papers.

**Good (chat small model):** roughly **< 20** is healthy; **~10–15** strong for tiny models.  
**Your ballpark:** val_loss≈3.06 → PPL≈**21**. OK for ~52M local model, not ChatGPT-level.

**Bad:** PPL >> vocab/10 with flat learning.

- [Perplexity explained](https://huggingface.co/docs/transformers/perplexity)
- [Jurafsky PPL notes](https://web.stanford.edu/~jurafsky/slp3/3.pdf)


In [ ]:
# Cell 6: perplexity from checkpoint + optional recompute on a few batches
ppl = float(math.exp(min(float(VAL_LOSS), 20.0))) if VAL_LOSS is not None else None
print(f"val_loss={VAL_LOSS}")
print(f"perplexity≈{ppl:.2f}" if ppl else "no val_loss")

# Quick recompute (subset) with same loss used in training
criterion = T.LabelSmoothing(size=vocab.get_vocab_size(), padding_idx=pad_idx, smoothing=0.1)
loss_fn = T.SimpleLossCompute(model.generator, criterion)
model.eval()
total_loss = 0.0
total_tok = 0
with torch.no_grad():
    for i, b in enumerate(valid_loader):
        batch = T.Batch(b[0], b[1], pad_idx)
        out = model.forward(batch.src, batch.tgt, batch.src_mask, batch.tgt_mask)
        loss, _ = loss_fn(out, batch.tgt_y, batch.ntokens)
        total_loss += float(loss)
        total_tok += int(batch.ntokens)
        if i + 1 >= MAX_ACC_BATCHES:
            break
mean_nll = total_loss / max(total_tok, 1)
re_ppl = math.exp(min(mean_nll, 20.0))
print(f"recomputed on {MAX_ACC_BATCHES} batches: mean_nll={mean_nll:.4f}  PPL≈{re_ppl:.2f}")
print("NOTE: training used label smoothing → recomputed NLL can differ slightly from logged val_loss.")

## 6 — Teacher-forced token accuracy

**What:** given true previous tokens, % of next-token argmax matches.  
**Why:** Fast proxy for “does the model know the distribution?” — **not** full chat quality.  
**Era:** Classic seq2seq (RNN→Transformer). Companies still log it in training dashboards.

**Good:** clearly above chance `1/vocab`. For chat, **~35–55%** is common for small models.  
**Bad:** near chance, or high TF-acc but terrible free decode (exposure bias).

- [Teacher forcing](https://machinelearningmastery.com/teacher-forcing-for-recurrent-neural-networks/)
- [Exposure bias](https://arxiv.org/abs/1511.06732)


In [ ]:
# Cell 7: token accuracy vs chance
acc, correct, total = T.token_accuracy(model, valid_loader, pad_idx, max_batches=MAX_ACC_BATCHES)
chance = 1.0 / max(vocab.get_vocab_size(), 1)
print(f"token_accuracy = {acc:.4f}  ({correct}/{total})")
print(f"chance (1/V)   = {chance:.6f}")
print(f"acc - chance   = {acc - chance:.4f}")
print("Read: higher is better, but this is NOT free-generation accuracy.")

## 7 — Free generation quality (overlap / exact-ish)

**What:** greedy decode vs gold reply → Jaccard word overlap, exact match rate.  
**Why:** Closer to real chat than teacher-forcing. Classic MT used BLEU; chat uses overlap + human prefs.  
**Era:** BLEU (2002) → ROUGE → now LLM-as-judge / preference (RLHF). Overlap is a cheap local proxy.

**Good:** mean overlap rising over training; some near-exact short replies.  
**Bad:** overlap ~0, or high overlap only because of short generic replies (“ok”, “yes”).

- [BLEU paper](https://aclanthology.org/P02-1040/)
- [Chat eval survey](https://arxiv.org/abs/2307.03109)


In [ ]:
# Cell 8: greedy decode vs expected (overlap table)
cmp = T.expected_vs_predicted_table(single_loader, model, vocab, n_examples=N_DECODE)
# clean helper columns
for col in ["expected", "predicted"]:
    if col in cmp.columns:
        cmp[col] = cmp[col].astype(str)

mean_ov = float(cmp["overlap"].mean()) if len(cmp) else 0.0
exact = int((cmp["overlap"] >= 0.999).sum()) if len(cmp) else 0
print(f"n={len(cmp)}  mean_overlap={mean_ov:.3f}  exactish={exact}/{len(cmp)}")
display(cmp.head(N_SHOW))

## 8 — Repetition & dullness (bad habits)

**What:** consecutive repeats, unique-token ratio, repeated bigrams.  
**Why:** Small models often loop (“I I I…”) or parrot safe phrases. Companies watch **repetition / diversity**.  
**Era:** Always a failure mode; strong focus since GPT-2/3 decoding papers.

**Good:** unique ratio high; few consecutive repeats.  
**Bad:** long loops, same 2–3 words dominate.

- [Nucleus sampling](https://arxiv.org/abs/1904.09751) (decoding fix, not training)
- [Degeneration](https://arxiv.org/abs/1904.09751)


In [ ]:
# Cell 9: repetition stats on decoded replies
SPECIAL = {"<s>", "</s>", "<blank>", "<unk>", ""}

def toks(text):
    return [t for t in str(text).replace("<s>", " ").replace("</s>", " ").split() if t not in SPECIAL]

rows = []
for _, r in cmp.iterrows():
    pt = toks(r["predicted"])
    if not pt:
        rows.append({"len": 0, "unique_ratio": 0, "max_run": 0, "bigram_repeat_rate": 0})
        continue
    # max consecutive run
    max_run, run = 1, 1
    for i in range(1, len(pt)):
        if pt[i] == pt[i-1]:
            run += 1
            max_run = max(max_run, run)
        else:
            run = 1
    bigrams = list(zip(pt, pt[1:]))
    if bigrams:
        c = collections.Counter(bigrams)
        bigram_rep = sum(v for v in c.values() if v > 1) / len(bigrams)
    else:
        bigram_rep = 0.0
    rows.append({
        "len": len(pt),
        "unique_ratio": len(set(pt)) / len(pt),
        "max_run": max_run,
        "bigram_repeat_rate": bigram_rep,
    })

rep = pd.DataFrame(rows)
print(rep.describe().round(3))
print("\nFlags:")
print("  mean unique_ratio (higher better):", round(float(rep.unique_ratio.mean()), 3))
print("  % replies with max_run>=3 (lower better):", round(100*float((rep.max_run>=3).mean()), 1), "%")
print("  mean bigram_repeat_rate (lower better):", round(float(rep.bigram_repeat_rate.mean()), 3))
print("  mean length:", round(float(rep.len.mean()), 1))

## 9 — Length mismatch

**What:** predicted words − expected words.  
**Why:** Models truncate, ramble, or ignore turn length. Useful with overlap.

**Good:** centered near 0.  
**Bad:** always very short (safe/generic) or always max_len loops.

- [Length bias in generation](https://arxiv.org/abs/1808.10006)


In [ ]:
# Cell 10: length bias
lens = []
for _, r in cmp.iterrows():
    e, p = toks(r["expected"]), toks(r["predicted"])
    lens.append({"exp": len(e), "pred": len(p), "diff": len(p)-len(e)})
L = pd.DataFrame(lens)
print(L.describe().round(2))
print("mean diff (pred-exp):", round(float(L["diff"].mean()), 2))

## 10 — Hallucination proxies (honest limits)

True **factuality** needs external knowledge / judges (big labs: TruthfulQA, FActScore, RAG checks).  
On this small chat model we use **cheap proxies**:

1. **Unsupported novelty** — many predicted words never in src∪gold (can be creative OR hallucinated).  
2. **Copy rate** — fraction of pred tokens also in source (high = safe copy; low = inventing).  
3. **Generic filler rate** — “ok / yes / i / you / …” dominance.

**Era:** Hallucination eval exploded with LLMs (2022+). For v4, treat these as **smoke tests**, not truth scores.

- [TruthfulQA](https://arxiv.org/abs/2109.07958)
- [FActScore](https://arxiv.org/abs/2305.14251)
- [Hallucination survey](https://arxiv.org/abs/2311.05232)


In [ ]:
# Cell 11: hallucination-ish proxies
FILLER = set("ok yes no yeah hi hello thanks thank you i you we the a an is are am to of and or but so just well maybe please".split())

hall_rows = []
for _, r in cmp.iterrows():
    src_t = set(toks(r["input"])) if "input" in cmp.columns else set()
    exp = set(toks(r["expected"]))
    pred = toks(r["predicted"])
    support = src_t | exp
    if pred:
        novel = [w for w in pred if w not in support]
        copy_from_exp = sum(1 for w in pred if w in exp) / len(pred)
        copy_from_src = sum(1 for w in pred if w in src_t) / len(pred) if src_t else 0.0
        filler = sum(1 for w in pred if w.lower() in FILLER) / len(pred)
        novel_rate = len(novel) / len(pred)
    else:
        novel_rate = copy_from_exp = copy_from_src = filler = 0.0
        novel = []
    hall_rows.append({
        "novel_rate": novel_rate,
        "copy_from_gold": copy_from_exp,
        "copy_from_src": copy_from_src,
        "filler_rate": filler,
        "n_novel": len(novel),
    })

H = pd.DataFrame(hall_rows)
print(H.describe().round(3))
print("\nHow to read (rough):")
print("  high filler_rate → dull / safe replies")
print("  high novel_rate with low overlap → inventing (may be hallucination OR valid paraphrase)")
print("  high copy_from_gold → sticking to gold vocabulary (safer, less creative)")
print("  high copy_from_src → echoing the user (can be good or lazy)")

## 11 — Qualitative spot checks (human eye)

Companies always keep a **fixed prompt suite** (safety, instruction, chitchat, math, refusal).  
Run a few prompts; judge: relevant? loops? invents facts?


In [ ]:
# Cell 12: fixed prompt suite
PROMPTS = [
    "Hi, how are you today?",
    "What is the capital of France?",
    "Write a short tip for sleeping better.",
    "My name is Alex. What is my name?",
    "Explain gravity in one sentence.",
    "Can you help me cook pasta?",
    "2 + 2 = ?",
    "Repeat the word banana three times.",
]

print("Greedy decode on fixed suite:\n")
for i, p in enumerate(PROMPTS):
    reply = T.generate_reply(
        model, vocab, spacy_en, p,
        max_len=int(CFG["max_padding"]),
        max_padding=int(CFG["max_padding"]),
    )
    print(f"[{i}] USER: {p}")
    print(f"    BOT : {reply}\n")

## 12 — Weight / training health

**What:** NaNs, extreme norms, dead-ish layers.  
**Why:** Bad hardware / LR / corrupted ckpt show up here before chat metrics.  
Labs monitor grad/weight norms during training; we spot-check the finished weights.

**Good:** finite tensors, reasonable embedding norms.  
**Bad:** NaN/Inf, all-zero rows, huge outliers.

- [PyTorch anomaly detection](https://pytorch.org/docs/stable/autograd.html#torch.autograd.detect_anomaly)


In [ ]:
# Cell 13: parameter health
stats = []
nan_params = 0
for name, p in model.named_parameters():
    t = p.detach().float().cpu()
    n_nan = int(torch.isnan(t).sum())
    n_inf = int(torch.isinf(t).sum())
    nan_params += n_nan + n_inf
    stats.append({
        "name": name,
        "numel": t.numel(),
        "mean": float(t.mean()),
        "std": float(t.std()),
        "absmax": float(t.abs().max()),
        "nan": n_nan,
        "inf": n_inf,
    })
S = pd.DataFrame(stats).sort_values("absmax", ascending=False)
print("total NaN/Inf elements:", nan_params)
display(S.head(12))
print("... lowest absmax ...")
display(S.tail(5))

## 13 — Company-style checklist (one by one)

Big labs (OpenAI / Google / Meta / Anthropic) typically layer checks like this. We run **local proxies** for each.

| # | Lab-style test | Our proxy | Why it matters |
|---|----------------|-----------|----------------|
| A | Training health | loss curve + weight NaNs | Did it learn / corrupt? |
| B | LM quality | PPL + TF token acc | Distribution fit |
| C | Task quality | overlap / exact | Usefulness of replies |
| D | Degeneration | repetition stats | User pain (loops) |
| E | Length control | length diff | Truncation / ramble |
| F | Faithfulness | novel/filler rates | Hallucination smell |
| G | Behavior suite | fixed prompts | Instruction / memory / facts |
| H | Capability ceiling | params + ctx + data mix | What hardware/data allow |

Links:
- [HELM](https://crfm.stanford.edu/helm/latest/)
- [BIG-bench](https://github.com/google/BIG-bench)
- [LM Evaluation Harness](https://github.com/EleutherAI/lm-evaluation-harness)
- [OpenAI evals](https://github.com/openai/evals)


In [ ]:
# Cell 14: checklist scorecard (auto-filled from metrics above)
scorecard = pd.DataFrame([
    {"test": "A training health", "signal": f"val_loss={VAL_LOSS}", "pass_hint": "falling val_loss, no NaNs"},
    {"test": "B LM quality", "signal": f"PPL≈{ppl:.1f}, TF-acc={acc:.3f}", "pass_hint": "PPL~<20 and TF-acc >> chance"},
    {"test": "C generation overlap", "signal": f"mean_overlap={mean_ov:.3f}", "pass_hint": "rising over epochs; not near 0"},
    {"test": "D repetition", "signal": f"unique={float(rep.unique_ratio.mean()):.3f}, run>=3={100*float((rep.max_run>=3).mean()):.1f}%", "pass_hint": "unique high, long runs rare"},
    {"test": "E length", "signal": f"mean_diff={float(L['diff'].mean()):.2f}", "pass_hint": "diff near 0"},
    {"test": "F hallucination proxies", "signal": f"novel={float(H.novel_rate.mean()):.3f}, filler={float(H.filler_rate.mean()):.3f}", "pass_hint": "not extreme; inspect prompts"},
    {"test": "G behavior suite", "signal": "see Cell 12 outputs", "pass_hint": "relevant, no loops, remembers name"},
    {"test": "H capacity", "signal": f"params={n_params:,}, ctx={CFG['max_padding']}, data={CFG.get('train_sources')}", "pass_hint": "more data/ctx/params usually help"},
])
display(scorecard)

## 14 — Summary: what matters **most now**

For a **local ~50M encoder–decoder chat model** (your v4), priority order today:

1. **Val loss / PPL + no degeneration** — if PPL stuck high or replies loop, nothing else matters.  
2. **Free-gen overlap / human spot checks** — TF-acc alone lies (exposure bias).  
3. **Repetition / filler** — biggest user-visible failure on small models.  
4. **Hallucination** — real factuality needs tools/RAG/bigger models; use proxies + suite only.  
5. **BLEU/ROUGE alone** — outdated as the *only* chat score; keep as secondary.

### Diagnose your bottleneck (typical for this setup)

| Area | Signs in *your* v4 | Likely? |
|------|--------------------|---------|
| **Hardware** | 4GB VRAM → small N/d_model/ctx | **Yes** — caps size & batch |
| **Data** | word-level tok; DailyDialog+Alpaca mix; short ctx 64 | **Yes** — tokenizer + mix/quality |
| **Architecture** | classic Enc–Dec (Annotated Transformer), not decoder-only LM | **Medium** — fine for dialogue pairs; GPT-style often better for open chat |
| **Training** | 5 epochs, loss still falling at end | **Medium** — maybe undertrained |

### What helps most for a better model (practical order)

1. **More / better data + BPE tokenizer** (biggest quality jump for small models).  
2. **Longer context + cleaner chat formatting** (roles, multi-turn).  
3. **Train longer / better schedule** if val_loss still dropping.  
4. **Bigger model on Kaggle/Colab** (more layers / d_model) when VRAM allows.  
5. **Better decoding** (temperature / nucleus) to cut loops — helps UX fast.  
6. **Later:** instruction finetune + preference (DPO/RLHF) once base LM is solid.  
7. **Compare** against your GPT-v1 decoder-only track for open-ended chat.

### Quick “is it good?” cheat sheet

| Metric | Weak | OK for local v4 | Strong |
|--------|------|-----------------|--------|
| PPL | >40 | ~15–25 | <12 |
| TF token acc | ~chance | clearly > chance | rising + stable |
| Overlap | ~0 | some structure | many close replies |
| max_run≥3 | common | rare | almost never |
| Name memory prompt | fails | sometimes | consistent |

Re-run this notebook after each training change; keep Cell 12 prompts **fixed** so you can compare versions fairly.


In [ ]:
# Cell 15: one-screen verdict blob
verdict = {
    "checkpoint": Path(CKPT_PATH).name,
    "val_loss": VAL_LOSS,
    "perplexity": None if ppl is None else round(ppl, 2),
    "token_accuracy": round(acc, 4),
    "chance": round(chance, 6),
    "mean_overlap": round(mean_ov, 3),
    "exactish": f"{exact}/{len(cmp)}",
    "unique_ratio_mean": round(float(rep.unique_ratio.mean()), 3),
    "pct_max_run_ge3": round(100*float((rep.max_run>=3).mean()), 1),
    "params_M": round(n_params/1e6, 2),
    "max_padding": CFG.get("max_padding"),
    "train_sources": CFG.get("train_sources"),
}
print("=== v4 EVAL SNAPSHOT ===")
for k, v in verdict.items():
    print(f"{k:22s} {v}")
print("\nDone. Scroll markdown summary above for diagnosis + upgrade order.")